In [ ]:
# Data manipulation and visualization
import pandas as pd
import joblib
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from scipy.stats import f_oneway

# Model
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Evaluation metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [ ]:
df = pd.read_csv("youtube_ad_revenue_dataset.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['date'].dtype

In [ ]:
for col in df:
    print(df[col].value_counts())
    print()

## Removing duplicated rows

In [ ]:
df = df.drop_duplicates()

In [ ]:
df[df.duplicated(keep='first')]

In [ ]:
df.describe()

In [ ]:
def outlier(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    upper_limit = Q3 + IQR
    lower_limit = Q1 - IQR
    return upper_limit, lower_limit

In [ ]:
df_out_list = df.drop(columns = ['video_id', 'date', 'category', 'device', 'country'])
df_out_list.columns

In [ ]:
for col in df_out_list:
    print(col, outlier(df[col]))

In [ ]:
df.isnull().sum()

## Handling null values

In [ ]:
df['likes'] = df['likes'].fillna(df['likes'].median())

In [ ]:
df['comments'] = df['comments'].fillna(df['comments'].median())

In [ ]:
df['watch_time_minutes'] = df['watch_time_minutes'].fillna(df['watch_time_minutes'].median())

In [ ]:
df['engagement_rate'] = (df['likes'] + df['comments']) / df['views']

In [ ]:
plt.boxplot(df['likes'])
plt.show()

In [ ]:
df.head()

In [ ]:
plt.boxplot(df['comments'])
plt.show()

In [ ]:
plt.boxplot(df['watch_time_minutes'].dropna())
plt.show()

## Pearson Correlation analysis to compare numerical columns with ad_revenue

In [ ]:
df_cor = df.drop(columns = ['video_id', 'category', 'device', 'country','date'])
df_cor.corr()

In [ ]:
corr = df_cor.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df['ad_revenue_usd'].skew()

## Visualization to check for skewness

In [ ]:
sns.histplot(df['ad_revenue_usd'], kde=True)
plt.show()

In [ ]:
sns.boxplot(x='category', y='ad_revenue_usd', data=df)

In [ ]:
sns.boxplot(x='device', y='ad_revenue_usd', data=df)

In [ ]:
sns.boxplot(x='country', y='ad_revenue_usd', data=df)

## ANOVA for categorical group with ad_revenue

In [ ]:
groups = [group['ad_revenue_usd'] for name, group in df.groupby('category')]

f_stat, p_value = f_oneway(*groups)

print('category column P_val',p_value)

In [ ]:
groups = [group['ad_revenue_usd'] for name, group in df.groupby('device')]

f_stat, p_value = f_oneway(*groups)

print('Device column P_val',p_value)

In [ ]:
groups = [group['ad_revenue_usd'] for name, group in df.groupby('country')]

f_stat, p_value = f_oneway(*groups)

print('Country column P_val',p_value)

In [ ]:
df['watch_time_minutes'].skew(), df['video_length_minutes'].skew()

## Preprocessing features

In [ ]:
Num_con_transformer = Pipeline(steps=[
  ('Imputer',SimpleImputer(strategy='median')),
  ('Scaler',StandardScaler()),
])

Cat_transformer = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoding',OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [ ]:
pre_process_steps= ColumnTransformer(transformers=[
    ('Num_continous_data',Num_con_transformer,['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers']),
    ('categorical_data',Cat_transformer,['category', 'device', 'country']),
])

## Linear Regression model

In [ ]:
LR_pipeline = Pipeline(steps=[
    ('Pre_Process',pre_process_steps),
    ('LR',LinearRegression())
])
LR_pipeline

In [ ]:
X = df[['views', 'likes', 'comments', 'watch_time_minutes','video_length_minutes', 'subscribers', 'category', 'device', 'country']]
y = df[['ad_revenue_usd']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [ ]:
LR_pipeline.fit(X_train,y_train)

In [ ]:
LR_pipeline.steps[1][1].intercept_

In [ ]:
LR_pipeline.steps[1][1].coef_

In [ ]:
feature_names = LR_pipeline.named_steps['Pre_Process'].get_feature_names_out()
coef_df = pd.DataFrame({
    'feature' : feature_names,
    'coefficient' : LR_pipeline.steps[1][1].coef_.ravel()
})
coef_df = coef_df.sort_values(by='coefficient', ascending=False)
coef_df[coef_df['feature'].str.contains('country')]

In [ ]:
actual_watch_time_coef = coef_df['coefficient'][3] / X_train['watch_time_minutes'].std()
actual_likes_coef = coef_df['coefficient'][1] / X_train['likes'].std()
actual_comments_coef = coef_df['coefficient'][2] / X_train['comments'].std()
actual_views_coef = coef_df['coefficient'][0] / X_train['views'].std()
actual_subscribers_coef = coef_df['coefficient'][5] / X_train['subscribers'].std()
actual_video_length_minutes_coef = coef_df['coefficient'][4] / X_train['video_length_minutes'].std()
print('actual_watch_time_coef',actual_watch_time_coef)
print('actual_likes_coef',actual_likes_coef)
print('actual_comments_coef',actual_comments_coef)
print('actual_views_coef',actual_views_coef)
print('actual_subscribers_coe',actual_subscribers_coef)
print('actual_video_length_minutes_coef',actual_video_length_minutes_coef)

In [ ]:
X_train['country'].unique()

In [ ]:
y_train_predict_LR = LR_pipeline.predict(X_train)
y_test_predict_LR = LR_pipeline.predict(X_test)

In [ ]:
pred = LR_pipeline.predict(X_test.iloc[[0]])
pred
X_test.iloc[[0]]

In [ ]:
MSE_Train = mean_squared_error(y_train,y_train_predict_LR)
MSE_Test = mean_squared_error(y_test,y_test_predict_LR)
print(f'MSE_Train: {MSE_Train}, RMSE_Train: {np.sqrt(MSE_Train)}')
print(f'MSE_Test: {MSE_Test}, RMSE_Test: {np.sqrt(MSE_Test)}')


In [ ]:
r2_train = r2_score(y_train,y_train_predict_LR)
r2_test = r2_score(y_test,y_test_predict_LR)
print('R2_train:',r2_train)
print('R2_test:',r2_test)

## Saving model as pickel file

In [ ]:
joblib.dump(LR_pipeline, 'ad_revenue_model.pkl')

## Polynomial regression model

In [ ]:
Polynomial_pipeline = Pipeline(steps=[
    ('Pre_Process',pre_process_steps),
    ('ploy', PolynomialFeatures(degree=3)),
    ('LR',LinearRegression())
])
Polynomial_pipeline

In [ ]:
Polynomial_pipeline.fit(X_train, y_train)

In [ ]:
y_train_predict_PL = LR_pipeline.predict(X_train)
y_test_predict_PL = LR_pipeline.predict(X_test)

In [ ]:
MSE_Train = mean_squared_error(y_train,y_train_predict_PL)
MSE_Test = mean_squared_error(y_test,y_test_predict_PL)
print(f'MSE_Train: {MSE_Train}, RMSE_Train: {np.sqrt(MSE_Train)}')
print(f'MSE_Test: {MSE_Test}, RMSE_Test: {np.sqrt(MSE_Test)}')
r2_train = r2_score(y_train,y_train_predict_PL)
r2_test = r2_score(y_test,y_test_predict_PL)
print('R2_train:',r2_train)
print('R2_test:',r2_test)

## Decision Tree

In [ ]:
DT_pipeline = Pipeline(steps=[
    ('Pre_Process', pre_process_steps),
    ('DT', DecisionTreeRegressor(max_depth=10,random_state=42))
])

DT_pipeline

In [ ]:
DT_pipeline.fit(X_train,y_train)

In [ ]:
y_train_predict_DT = DT_pipeline.predict(X_train)
y_test_predict_DT = DT_pipeline.predict(X_test)

In [ ]:
MSE_Train = mean_squared_error(y_train,y_train_predict_DT)
MSE_Test = mean_squared_error(y_test,y_test_predict_DT)
print(f'MSE_Train: {MSE_Train}, RMSE_Train: {np.sqrt(MSE_Train)}')
print(f'MSE_Test: {MSE_Test}, RMSE_Test: {np.sqrt(MSE_Test)}')
r2_train = r2_score(y_train,y_train_predict_DT)
r2_test = r2_score(y_test,y_test_predict_DT)
print('R2_train:',r2_train)
print('R2_test:',r2_test)

## Random Forest

In [ ]:
RF_pipeline = Pipeline(steps=[
    ('Pre_Process',pre_process_steps),
    ('RF',RandomForestRegressor(n_estimators=10,max_depth=10,random_state=42))
])

In [ ]:
RF_pipeline.fit(X_train,y_train)

In [ ]:
y_train_predict_RF = RF_pipeline.predict(X_train)
y_test_predict_RF = RF_pipeline.predict(X_test)

In [ ]:
MSE_Train = mean_squared_error(y_train,y_train_predict_RF)
MSE_Test = mean_squared_error(y_test,y_test_predict_RF)
print(f'MSE_Train: {MSE_Train}, RMSE_Train: {np.sqrt(MSE_Train)}')
print(f'MSE_Test: {MSE_Test}, RMSE_Test: {np.sqrt(MSE_Test)}')
r2_train = r2_score(y_train,y_train_predict_RF)
r2_test = r2_score(y_test,y_test_predict_RF)
print('R2_train:',r2_train)
print('R2_test:',r2_test)

## XG Boost

In [ ]:
XGB_pipeline = Pipeline(steps=[
    ('pre_process',pre_process_steps),
    ('XGB', XGBRegressor(n_estimators=10,max_depth=10,random_state=42))
])

In [ ]:
XGB_pipeline.fit(X_train, y_train)

In [ ]:
y_train_predict_XGB = XGB_pipeline.predict(X_train)
y_test_predict_XGB = XGB_pipeline.predict(X_test)

In [ ]:
MSE_Train = mean_squared_error(y_train,y_train_predict_XGB)
MSE_Test = mean_squared_error(y_test,y_test_predict_XGB)
print(f'MSE_Train: {MSE_Train}, RMSE_Train: {np.sqrt(MSE_Train)}')
print(f'MSE_Test: {MSE_Test}, RMSE_Test: {np.sqrt(MSE_Test)}')
r2_train = r2_score(y_train,y_train_predict_XGB)
r2_test = r2_score(y_test,y_test_predict_XGB)
print('R2_train:',r2_train)
print('R2_test:',r2_test)